In [3]:
# 01_bronze_ingest - Cell 1: imports and constants
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
LAKEHOUSE_ROOT = "/lakehouse/default/Files/bronze/yellow_taxi"
YEARS = [2022, 2023, 2024]
MONTHS = list(range(1, 13))

print(f"Will download {len(YEARS) * len(MONTHS)} files: {YEARS[0]}-{YEARS[-1]}")

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 5, Finished, Available, Finished, False)

Will download 36 files: 2022-2024


In [4]:
# 01_bronze_ingest - Cell 2: download function with idempotency

def download_month(year: int, month: int) -> dict:
    """Download one monthly parquet from TLC CloudFront. Idempotent."""
    fname = f"yellow_tripdata_{year}-{month:02d}.parquet"
    url   = f"{BASE_URL}/{fname}"
    out_d = f"{LAKEHOUSE_ROOT}/year={year}/month={month:02d}"
    out_f = f"{out_d}/data.parquet"

    if os.path.exists(out_f) and os.path.getsize(out_f) > 0:
        return {
            "year": year, "month": month, "status": "skipped",
            "size_mb": round(os.path.getsize(out_f) / 1024 / 1024, 1)
        }

    os.makedirs(out_d, exist_ok=True)
    t0 = time.time()

    try:
        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(out_f, "wb") as f:
                for chunk in r.iter_content(1024 * 1024):
                    f.write(chunk)
        return {
            "year": year, "month": month, "status": "downloaded",
            "size_mb": round(os.path.getsize(out_f) / 1024 / 1024, 1),
            "duration_sec": round(time.time() - t0, 1)
        }
    except Exception as e:
        return {"year": year, "month": month, "status": "failed", "error": str(e)}

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 6, Finished, Available, Finished, False)

In [5]:
# 01_bronze_ingest - Cell 3: parallel execution with logging
tasks   = [(y, m) for y in YEARS for m in MONTHS]
results = []

print(f"Starting download of {len(tasks)} files (3 workers)...\n")
t_start = time.time()

with ThreadPoolExecutor(max_workers=3) as executor:
    for r in executor.map(lambda t: download_month(*t), tasks):
        results.append(r)
        tag = f"{r['year']}-{r['month']:02d}"
        if r['status'] == 'downloaded':
            print(f"  {tag}  downloaded  {r['size_mb']:>6.1f} MB  {r['duration_sec']:>5.1f}s")
        elif r['status'] == 'skipped':
            print(f"  {tag}  skipped     {r['size_mb']:>6.1f} MB  (already on disk)")
        else:
            print(f"  {tag}  FAILED      {r.get('error', 'unknown')}")

ok       = sum(1 for r in results if r['status'] in ('downloaded', 'skipped'))
total_mb = sum(r['size_mb'] for r in results if 'size_mb' in r)
print(f"\nDone: {ok}/{len(tasks)} files OK, {total_mb:.0f} MB total, {time.time()-t_start:.0f}s elapsed.")

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 7, Finished, Available, Finished, False)

Starting download of 36 files (3 workers)...

  2022-01  skipped       36.4 MB  (already on disk)
  2022-02  skipped       43.5 MB  (already on disk)
  2022-03  skipped       53.1 MB  (already on disk)
  2022-04  skipped       52.7 MB  (already on disk)
  2022-05  skipped       53.0 MB  (already on disk)
  2022-06  skipped       52.8 MB  (already on disk)
  2022-07  skipped       47.1 MB  (already on disk)
  2022-08  skipped       47.4 MB  (already on disk)
  2022-09  skipped       47.3 MB  (already on disk)
  2022-10  skipped       54.4 MB  (already on disk)
  2022-11  skipped       47.8 MB  (already on disk)
  2022-12  skipped       51.2 MB  (already on disk)
  2023-01  skipped       45.5 MB  (already on disk)
  2023-02  skipped       45.5 MB  (already on disk)
  2023-03  skipped       53.5 MB  (already on disk)
  2023-04  skipped       51.7 MB  (already on disk)
  2023-05  skipped       55.9 MB  (already on disk)
  2023-06  skipped       52.5 MB  (already on disk)
  2023-07  skipped

In [6]:
# 01_bronze_ingest - Cell 4 (FIXED): handle TLC schema drift
from pyspark.sql import functions as F
from functools import reduce

# Canonical schema — what we want regardless of source file variation.
# Numeric IDs as BIGINT, monetary as DOUBLE, code-like fields as DOUBLE.
TARGET_COLS = [
    ("VendorID",              "bigint"),
    ("tpep_pickup_datetime",  "timestamp"),
    ("tpep_dropoff_datetime", "timestamp"),
    ("passenger_count",       "double"),
    ("trip_distance",         "double"),
    ("RatecodeID",            "double"),
    ("store_and_fwd_flag",    "string"),
    ("PULocationID",          "bigint"),
    ("DOLocationID",          "bigint"),
    ("payment_type",          "bigint"),
    ("fare_amount",           "double"),
    ("extra",                 "double"),
    ("mta_tax",               "double"),
    ("tip_amount",            "double"),
    ("tolls_amount",          "double"),
    ("improvement_surcharge", "double"),
    ("total_amount",          "double"),
    ("congestion_surcharge",  "double"),
    ("airport_fee",           "double"),
]

def read_normalized(year: int, month: int):
    """Read one monthly parquet and normalize to the canonical schema.
       Handles: case-insensitive column names + type casting + missing columns."""
    path = f"Files/bronze/yellow_taxi/year={year}/month={month:02d}/data.parquet"
    df = spark.read.parquet(path)
    # Case-insensitive lookup of source column names
    col_lookup = {c.lower(): c for c in df.columns}
    select_exprs = []
    for tgt_name, tgt_type in TARGET_COLS:
        src_name = col_lookup.get(tgt_name.lower())
        if src_name is None:
            # Column missing in this file — emit NULL with the target type
            select_exprs.append(F.lit(None).cast(tgt_type).alias(tgt_name))
        else:
            select_exprs.append(F.col(src_name).cast(tgt_type).alias(tgt_name))
    return (df.select(*select_exprs)
              .withColumn("ingest_year",  F.lit(year))
              .withColumn("ingest_month", F.lit(month)))

print("Reading and normalizing 36 monthly files...")
dfs = [read_normalized(y, m) for y in YEARS for m in MONTHS]
df_bronze = reduce(lambda a, b: a.unionByName(b), dfs)

row_count = df_bronze.count()
print(f"\nTotal rows: {row_count:,}")
print(f"Total columns: {len(df_bronze.columns)}")
print("\nSchema:")
df_bronze.printSchema()



StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 8, Finished, Available, Finished, False)

Reading and normalizing 36 monthly files...

Total rows: 119,136,044
Total columns: 21

Schema:
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- ingest_year: integer (nullable = false

AttributeError: 'DataFrame' object has no attribute 'display'

In [8]:
print("\nSample (5 rows):")
display(df_bronze.limit(5))

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 10, Finished, Available, Finished, False)


Sample (5 rows):


SynapseWidget(Synapse.DataFrame, 9fb49c7e-e3e8-4119-be1c-9ede8428709d)

In [9]:
# 01_bronze_ingest - Cell 5: write as Delta managed table in the Lakehouse
(df_bronze
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_yellow_trips_raw"))

print("Table bronze_yellow_trips_raw persisted.")
print(f"Lakehouse path: Tables/bronze_yellow_trips_raw")

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 11, Finished, Available, Finished, False)

Table bronze_yellow_trips_raw persisted.
Lakehouse path: Tables/bronze_yellow_trips_raw


In [10]:
# 01_bronze_ingest - Cell 6: ingest taxi zone lookup
import urllib.request

LOOKUP_DIR = "/lakehouse/default/Files/bronze/lookups"
os.makedirs(LOOKUP_DIR, exist_ok=True)

zone_url  = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_path = f"{LOOKUP_DIR}/taxi_zone_lookup.csv"
urllib.request.urlretrieve(zone_url, zone_path)
print(f"Downloaded: {zone_path}")

df_zones = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("Files/bronze/lookups/taxi_zone_lookup.csv")
)

print(f"\nTaxi zones loaded: {df_zones.count()} rows")
df_zones.show(5, truncate=False)

(df_zones
    .write
    .mode("overwrite")
    .saveAsTable("bronze_taxi_zone_lookup"))

print("\nTable bronze_taxi_zone_lookup persisted.")

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 12, Finished, Available, Finished, False)

Downloaded: /lakehouse/default/Files/bronze/lookups/taxi_zone_lookup.csv

Taxi zones loaded: 265 rows
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


Table bronze_taxi_zone_lookup persisted.


In [11]:
# 01_bronze_ingest - Cell 7: validation queries
from pyspark.sql import functions as F

print("===== Row count by year/month =====")
(spark.read.table("bronze_yellow_trips_raw")
    .groupBy(
        F.year("tpep_pickup_datetime").alias("year"),
        F.month("tpep_pickup_datetime").alias("month")
    )
    .count()
    .orderBy("year", "month")
    .show(40))

print("\n===== Date range =====")
(spark.read.table("bronze_yellow_trips_raw")
    .agg(
        F.min("tpep_pickup_datetime").alias("min_pickup"),
        F.max("tpep_pickup_datetime").alias("max_pickup")
    )
    .show(truncate=False))

print("\n===== Lookup sample =====")
spark.read.table("bronze_taxi_zone_lookup").show(5, truncate=False)

StatementMeta(, 16998b8c-ec57-433f-9fa4-4adb37481e34, 13, Finished, Available, Finished, False)

===== Row count by year/month =====
+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2001|    1|     11|
|2001|    8|      1|
|2002|   10|    439|
|2002|   12|     24|
|2003|    1|     15|
|2008|   12|     67|
|2009|    1|     73|
|2012|    2|      1|
|2014|   11|      1|
|2021|   12|     24|
|2022|    1|2463900|
|2022|    2|2979419|
|2022|    3|3627888|
|2022|    4|3599902|
|2022|    5|3588292|
|2022|    6|3557692|
|2022|    7|3174363|
|2022|    8|3152694|
|2022|    9|3183784|
|2022|   10|3675384|
|2022|   11|3252670|
|2022|   12|3399585|
|2023|    1|3066731|
|2023|    2|2914003|
|2023|    3|3403660|
|2023|    4|3288249|
|2023|    5|3513664|
|2023|    6|3307259|
|2023|    7|2907093|
|2023|    8|2824201|
|2023|    9|2846741|
|2023|   10|3522269|
|2023|   11|3339731|
|2023|   12|3376537|
|2024|    1|2964623|
|2024|    2|3007533|
|2024|    3|3582611|
|2024|    4|3514295|
|2024|    5|3723843|
|2024|    6|3539170|
+----+-----+-------+
only showing top 40 rows


===== Date ra